# **SETUP**

In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# Load env variables
load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

Bro API KEY Variable exists


# **PARALLEL CHAINS**

In [2]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief: {input}")
])

In [3]:
# TASK - 2 [LLM]

llm_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

In [4]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

# **CUSTOM RUNNABLE**

In [5]:
# TASK - 4 [Custom Runnable]

from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:

    return {"text": text}

dictionary_maker_runnable = RunnableLambda(
    dictionary_maker
)

# **PARALLEL CHAIN - 1**

In [6]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    (
        "human",
        "Create a post for the following text for LinkedIn: {text}"
    )
])

# TASK - 2 [LLM]

llm_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

chain_linkedin = (
    linkedin_prompt
    | llm_groq
    | str_parser
)

# **PARALLEL CHAIN - 2**

In [7]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnableLambda
)

In [9]:
def insta_chain(text: dict):

    text = text["text"]

    # TASK - 1 [Prompt]

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        (
            "human",
            "Create a post for the following text for Instagram: {text}"
        )
    ])

    # TASK - 2 [LLM]

    llm_groq = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.7
    )

    # TASK - 3 [Str Parser]

    str_parser = StrOutputParser()

    chain_insta = (
        insta_prompt
        | llm_groq
        | str_parser
    )

    result = chain_insta.invoke({"text": text})

    return result

insta_chain_runnable = RunnableLambda(
    insta_chain
)

# **FINAL ORCHESTRATION**

In [10]:
final_chain = (
    prompt_template
    | llm_groq
    | str_parser
    | dictionary_maker_runnable
    | RunnableParallel(
        branches={
            "linkedin": chain_linkedin,
            "instagram": insta_chain_runnable
        }
    )
)

In [11]:
final_chain.invoke("KGF")

{'branches': {'linkedin': "Here's a LinkedIn-style post based on the provided text:\n\n**Leadership Lessons from the Big Screen: The Unstoppable Rise of Raja Krishnappa Bairya**\n\nAs professionals, we often draw inspiration from unexpected places. For me, the 2018 Indian action film KGF is a powerful reminder of the importance of perseverance, loyalty, and fighting for what's right.\n\nThe movie tells the story of Raja Krishnappa Bairya, a young man from a humble background who rises to become a dominant force in the Kolar Gold Fields. His journey is a testament to the human spirit's capacity for growth, adaptation, and transformation.\n\nAs we navigate our own careers and industries, we can learn valuable lessons from Raja's story:\n\n **Courage in the face of adversity**: Raja's bravery in the face of ruthless opposition is a reminder that we must stand up for what we believe in, even when the odds are against us.\n\n **The power of loyalty**: Raja's commitment to his community and 

# **BEAUTIFY OUTPUT**

In [12]:
# TASK - 1 [Beautify Function]

def beautify(final_response: dict) -> dict:

    linkedin_response = final_response['branches']['linkedin']

    instagram_response = final_response['branches']['instagram']

    return {
        "linkedin": linkedin_response,
        "instagram": instagram_response
    }

beautify_runnable = RunnableLambda(
    beautify
)

# Beautified Chain

beautified_chain = (
    final_chain
    | beautify_runnable
)

beautified_chain.invoke("Pushpa")

{'linkedin': 'Here\'s a LinkedIn-style post based on the provided text:\n\n**Lessons from the Big Screen: Leadership and Perseverance in the Face of Adversity**\n\nAs professionals, we often draw inspiration from unexpected sources. For me, the 2021 Indian film "Pushpa: The Rise" is a powerful reminder of the importance of resilience, strategic thinking, and leadership in overcoming obstacles.\n\nThe movie\'s protagonist, Pushpa Raj, is a testament to the human spirit\'s capacity for growth and transformation. From humble beginnings as a laborer, he rises to become a key player in the red sandalwood smuggling business, navigating treacherous rivalries, corruption, and personal struggles along the way.\n\nWhat can we learn from Pushpa\'s journey?\n\n **Adaptability is key**: Pushpa\'s ability to pivot and adjust to changing circumstances is a crucial factor in his success.\n **Loyalty and trust are essential**: The bonds he forms with his allies and the loyalty he inspires in others are